In [43]:
import pandas as pd

In [44]:
train_df = pd.read_csv("../../comet-commonsense/train_english_atomic.csv")
test_df = pd.read_csv('../../comet-commonsense/test_english_atomic.csv')
val_df = pd.read_csv('../../comet-commonsense/val_english_atomic.csv')

In [45]:
input = train_df["input"].iloc[2]
x_react = eval(train_df["xReact"].iloc[2])
x_want = eval(train_df["xWant"].iloc[2])
x_need = eval(train_df["xNeed"].iloc[2])
x_intent = eval(train_df["xIntent"].iloc[2])
x_effect = eval(train_df["xEffect"].iloc[2])
print(x_react["beams"][0])
print(x_want["beams"][0])
print(x_need["beams"][0])
print(x_intent["beams"][0])
print(x_effect["beams"][0])

sad
to apologize
none
none
none


In [46]:

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L12-v2',device="cpu")

In [47]:
import json
 
filename = './nodes/affective_states.txt'
with open(filename) as f:
    affective_states = f.read()
    affective_states = json.loads(affective_states)
    # print(affective_states)
print(affective_states["nodes"][0])
print(len(affective_states["nodes"]))

{'id': 'e28-t1', 'label': 'Sad', 'value': 1937}
22037


In [48]:
filename = './nodes/expectations.txt'
with open(filename) as f:
    expectations = f.read()
    expectations = json.loads(expectations)
    # print(expectations)
print(expectations["nodes"][0])

{'id': 'i1', 'label': 'What do I do about it?', 'value': 489}


In [49]:
filename = "./nodes/feedback.txt"
with open(filename) as f:
    feedback = f.read()
    feedback = json.loads(feedback)
    # print(feedback)
print(feedback["nodes"][0])

{'id': 'f1', 'label': 'thanks.', 'value': 960}


In [50]:
filename = "./nodes/responses.txt"
with open(filename) as f:
    responses = f.read()
    responses = json.loads(responses)
    print(responses)
print(responses["nodes"][0])

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [51]:
filename = "./nodes/stressors.txt"
with open(filename) as f:
    stressors = f.read()
    stressors = json.loads(stressors)
    # print(stressors)
print(stressors["nodes"][0])

{'id': 't1', 'label': 'commit, killing, death, painless, option', 'value': 11856, 'story': "|considering suicide|\nWhy does anything matter?\nWhat's the purpose of life?\nWhat should I do next, and why would it matter?\nMake the world a better place?\nOnly if I cared about its inhabitants.\nAnd anyway, none of that will matter when I'm gone... Why should I care about it?"}


In [52]:
filename = "./edges/affective_states.txt"
with open(filename) as f:
    affective_states_edges_s = f.read()
    affective_states_edges_s = json.loads(affective_states_edges_s)
    # print(affective_states_edges_s)
print(affective_states_edges_s["edges"][0])
# from stressors to affective states

{'from': 't1', 'to': 'e28-t1', 'value': 1937}


In [53]:
filename = "./edges/expectations-responses.txt"
with open(filename) as f:
    expectations_edges_r = f.read()
    expectations_edges_r = json.loads(expectations_edges_r)
    # print(expectations_edges_r)
print(expectations_edges_r["edges"][0])
# from expectations to responses

{'from': 'i244', 'to': 'r1', 'value': 1}


In [54]:
filename = "./edges/expectations-stressors.txt"
with open(filename) as f:
    expectations_edges_s = f.read()
    expectations_edges_s = json.loads(expectations_edges_s)
    # print(expectations_edges_s)
print(expectations_edges_s["edges"][0])
# from stressors to expectations

{'from': 't1', 'to': 'i1', 'value': 103}


In [55]:
filename = "./edges/responses-feedback.txt"
with open(filename) as f:
    responses_edges_f = f.read()
    responses_edges_f = json.loads(responses_edges_f)
    # print(responses_edges_f)
print(responses_edges_f["edges"][0])
# from responses to feedback

{'from': 'r210', 'to': 'f1', 'value': 3}


In [56]:
filename = "./edges/stressors-responses.txt"
with open(filename) as f:
    stressors_edges_r = f.read()
    stressors_edges_r = json.loads(stressors_edges_r)
    # print(stressors_edges_r)
print(stressors_edges_r["edges"][0])
# from stressors to responses

{'from': 't1', 'to': 'r1', 'value': 273}


In [57]:
print(x_react["beams"][0])
print(x_want["beams"][0])
print(x_need["beams"][0])
print(x_intent["beams"][0])
print(x_effect["beams"][0])

sad
to apologize
none
none
none


In [58]:
def get_nodes(input,all_nodes,k):
    sentences = []
    for i in range(len(all_nodes)):
        sentences.append(all_nodes[i]["label"])
    embedding_input = model.encode(input, convert_to_tensor=False)
    embeddings = model.encode(sentences, convert_to_tensor=False)
    # print(embeddings.shape)
    # print(embedding_input.shape)
    cosine_scores = util.pytorch_cos_sim(embedding_input, embeddings)
    # print(cosine_scores)
    # print(cosine_scores[0][0].item())
    for i in range(len(all_nodes)):
        all_nodes[i]["score"] = cosine_scores[0][i].item()
    all_nodes.sort(key=lambda x: x["score"], reverse=True)
    return all_nodes[0:k]

In [59]:
def find_edges(node,edges,check_nodes):
    final_node = None
    simialirty_score = 0
    for i in range(len(edges)):
        if edges[i]["from"] == node["id"] or edges[i]["to"] == node["id"]:
            to_id = edges[i]["to"]
            from_id = edges[i]["from"]
            # print(to_id)
            for j in range(len(check_nodes)):
                if check_nodes[j]["id"] == to_id or check_nodes[j]["id"] == from_id:
                    if check_nodes[j]["score"] > simialirty_score:
                        simialirty_score = check_nodes[j]["score"]
                        final_node = check_nodes[j]
    return final_node

In [60]:
expectation_nodes = get_nodes(input,expectations["nodes"],1)
print(input)
print(expectation_nodes)
stressor_nodes = get_nodes(x_intent["beams"][0],stressors["nodes"],1)
print(x_intent["beams"][0])
print(stressor_nodes)
affective_state_nodes = get_nodes(x_react["beams"][0],affective_states["nodes"],1)
print(x_react["beams"][0])
print(affective_state_nodes)
response_x = x_want["beams"][0] + " , " + x_need["beams"][0] + " , " + x_effect["beams"][0]
response_nodes = get_nodes(response_x,responses["nodes"],1)
print(response_x)
print(response_nodes)

stressor_node = stressor_nodes[0]
print(stressor_node)
expectations_connection = find_edges(stressor_node,expectations_edges_s["edges"],expectations["nodes"])
print(expectations_connection)
response_connection = find_edges(stressor_node,stressors_edges_r["edges"],responses["nodes"])
print(response_connection)
affective_state_connection = find_edges(stressor_node,affective_states_edges_s["edges"],affective_states["nodes"])
print(affective_state_connection)


So looking into one I'm going to make a mistake. That kind o
[{'id': 'i2123', 'label': 'Is there a certain kind I should be looking for?', 'value': 2, 'score': 0.5556679964065552}]
none
[{'id': 't82', 'label': 'sex, drive, bed, sexual, zero', 'value': 79, 'story': "|What's your most wackiest side effect of depression med?|\nI used ti take Prozac, which drove my sex drive crazy.\nNow I switched to Lexapro, I have near zero sex drive.\nHows yours?", 'score': 0.3755347728729248}]
sad
[{'id': 'e28-t1', 'label': 'Sad', 'value': 1937, 'score': 1.0000007152557373}]
to apologize , none , none
[{'id': 'r4159', 'label': 'stop apologising for nothing.', 'labels': ['Stop apologising and reach out to someone.', 'stop apologising for nothing.', 'Stop apologizing.', 'STOP SAYING SORRY.'], 'value': 4, 'score': 0.6555059552192688}]
{'id': 't82', 'label': 'sex, drive, bed, sexual, zero', 'value': 79, 'story': "|What's your most wackiest side effect of depression med?|\nI used ti take Prozac, which drove

In [67]:
print(input)

stressor_node = stressor_nodes[0]
print(stressor_node)
expectations_connection = find_edges(stressor_node,expectations_edges_s["edges"],expectations["nodes"])
print(expectations_connection)
response_connection = find_edges(stressor_node,stressors_edges_r["edges"],responses["nodes"])
print(response_connection)
affective_state_connection = find_edges(stressor_node,affective_states_edges_s["edges"],affective_states["nodes"])
print(affective_state_connection)

So looking into one I'm going to make a mistake. That kind o
{'id': 't82', 'label': 'sex, drive, bed, sexual, zero', 'value': 79, 'story': "|What's your most wackiest side effect of depression med?|\nI used ti take Prozac, which drove my sex drive crazy.\nNow I switched to Lexapro, I have near zero sex drive.\nHows yours?", 'score': 0.3755347728729248}
{'id': 'i741', 'label': 'Anybody else have sex drive problems with those?', 'value': 4, 'score': 0.24270306527614594}
{'id': 'r308', 'label': 'Painfully honest.', 'labels': ['Best to be honest to yourself.', "Well, at least you're honest.", 'Just be honest.', 'Be honest.', 'Be very, very honest.', 'Be honest with them.', 'Be honest to yourself.', 'Be honest.', 'I am being honest here.', 'Be honest and open.', 'Be *honest*.', 'Honest question.', 'The most important thing is to be honest.', 'But keep yourself honest!', 'Be honest, no matter what.', 'Be honest.', 'Painfully honest.', 'Just be honest.', 'Perhaps, but honest.', 'You be honest

In [66]:
exp_node = expectation_nodes[0]
print(exp_node)
str_node = find_edges(exp_node,expectations_edges_s["edges"],stressors["nodes"])
print(str_node)
affective_state_connection = find_edges(str_node,affective_states_edges_s["edges"],affective_states["nodes"])
print(affective_state_connection)
response = find_edges(exp_node,expectations_edges_r["edges"],responses["nodes"])
print(response)

{'id': 'i2123', 'label': 'Is there a certain kind I should be looking for?', 'value': 2, 'score': 0.5556679964065552}
{'id': 't131', 'label': 'therapist, finding, therapists, seeing, process', 'value': 86, 'story': '|I finally made an appointment to talk to a therapist.|\nI really hope that things will get better soon, and that I can enjoy life.\nEDIT: thank you all for the support and encouraging advice!\nIt truly means the world to me.', 'score': 0.16408362984657288}
{'id': 'e6-t131', 'label': 'Apprehensive', 'value': 40, 'score': 0.2578401565551758}
{'id': 'r11998', 'label': 'Things were a lot better.', 'labels': ['Things were a lot better.', 'That worked out much better.'], 'value': 2, 'score': 0.16006533801555634}


{'id': 'i2123', 'label': 'Is there a certain kind I should be looking for?', 'value': 2, 'score': 0.5556679964065552}
{'id': 't131', 'label': 'therapist, finding, therapists, seeing, process', 'value': 86, 'story': '|I finally made an appointment to talk to a therapist.|\nI really hope that things will get better soon, and that I can enjoy life.\nEDIT: thank you all for the support and encouraging advice!\nIt truly means the world to me.', 'score': 0.16408362984657288}
{'id': 'e6-t131', 'label': 'Apprehensive', 'value': 40, 'score': 0.2578401565551758}


In [63]:
input = train_df["input"].iloc[2]
x_react = eval(train_df["xReact"].iloc[2])
x_want = eval(train_df["xWant"].iloc[2])
x_need = eval(train_df["xNeed"].iloc[2])
x_intent = eval(train_df["xIntent"].iloc[2])
x_effect = eval(train_df["xEffect"].iloc[2])
print(x_react["beams"][0])
print(x_want["beams"][0])
print(x_need["beams"][0])
print(x_intent["beams"][0])
print(x_effect["beams"][0])

sad
to apologize
none
none
none


In [68]:
train_df = pd.read_csv("../../comet-commonsense/train_english_atomic.csv")
test_df = pd.read_csv('../../comet-commonsense/test_english_atomic.csv')
val_df = pd.read_csv('../../comet-commonsense/val_english_atomic.csv')

In [72]:
train = train_df.to_dict('list')
test = test_df.to_dict('list')
val = val_df.to_dict('list')

In [74]:
train.keys()

dict_keys(['utterances', 'label', 'speaker', 'oEffect', 'oReact', 'oWant', 'xAttr', 'xEffect', 'xIntent', 'xNeed', 'xReact', 'xWant', 'input'])

In [87]:
def update_data(train):
    train["expectation"] = []
    train["stressor"] = []
    train["affective_state"] = []
    train["response"] = []
    for i in range(len(train["utterances"])):
        input = train["input"][i]
        x_react = eval(train["xReact"][i])
        x_want = eval(train["xWant"][i])
        x_need = eval(train["xNeed"][i])
        x_intent = eval(train["xIntent"][i])
        x_effect = eval(train["xEffect"][i])
        print(input)
        # print(x_react["beams"][0])
        # print(x_want["beams"][0])
        # print(x_need["beams"][0])
        # print(x_intent["beams"][0])
        # print(x_effect["beams"][0])
        stressor_node_x = None
        # print(x_intent["beams"])
        affective = None
        want = None
        need = None
        effect = None
        for j in range(3):
            if x_intent["beams"][j].lower() != "none" and stressor_node_x == None:
                stressor_node_x = x_intent["beams"][j]
            if x_react["beams"][j].lower() != "none" and affective == None:
                affective = x_react["beams"][j]
            if x_want["beams"][j].lower() != "none" and want == None:
                want = x_want["beams"][j]
            if x_need["beams"][j].lower() != "none" and need == None:
                need = x_need["beams"][j]
            if x_effect["beams"][j].lower() != "none" and effect == None:
                effect = x_effect["beams"][j] 
        print(stressor_node_x,affective,need,want,effect)
        if stressor_node_x == None or affective == None or (want is None and need is None and effect is None):
            train["expectation"].append({})
            train["stressor"].append({})
            train["affective_state"].append({})
            train["response"].append({})
            print("adding no knowledge")
        else:
            stressor_node = stressor_node_x
            expectation_nodes = get_nodes(input,expectations["nodes"],1)
            stressor_nodes = get_nodes(stressor_node,stressors["nodes"],1)
            affective_state_nodes = get_nodes(x_react["beams"][0],affective_states["nodes"],1)
            if want == None:
                want = "none"
            if need == None:
                need = "none"
            if effect == None:
                effect = "none"
            response_x = str(want) + " , " + str(need) + " , " + str(effect)
            response_nodes = get_nodes(response_x,responses["nodes"],1)
            print("fetched nodes")
            stressor_node = stressor_nodes[0]
            expectations_connection = find_edges(stressor_node,expectations_edges_s["edges"],expectations["nodes"])
            response_connection = find_edges(stressor_node,stressors_edges_r["edges"],responses["nodes"])
            affective_state_connection = find_edges(stressor_node,affective_states_edges_s["edges"],affective_states["nodes"])
            train["expectation"].append(expectations_connection)
            train["stressor"].append(stressor_node)
            train["affective_state"].append(affective_state_connection)
            train["response"].append(response_connection)
    return train
#         expectation_nodes = get_nodes(input,expectations["nodes"],1)
#         print(input)
#         print(expectation_nodes)
#         stressor_nodes = get_nodes(x_intent["beams"][0],stressors["nodes"],1)
#         print(x_intent["beams"][0])
#         print(stressor_nodes)
#         affective_state_nodes = get_nodes(x_react["beams"][0],affective_states["nodes"],1)
#         print(x_react["beams"][0])
#         print(affective_state_nodes)
#         response_x = x_want["beams"][0] + " , " + x_need["beams"][0] + " , " + x_effect["beams"][0]
#         response_nodes = get_nodes(response_x,responses["nodes"],1)
#         print(response_x)
#         print(response_nodes)

#         stressor_node = stressor_nodes[0]
#         print(stressor_node)
#         expectations_connection = find_edges(stressor_node,expectations_edges_s["edges"],expectations["nodes"])
#         print(expectations_connection)
#         response_connection = find_edges(stressor_node,stressors_edges_r["edges"],responses["nodes"])
#         print(response_connection)
#         affective_state_connection = find_edges(stressor_node,affective_states_edges_s["edges"],affective_states["nodes"])
#         print(affective_state_connection)
        

In [ ]:
print("Starting train")
new_train = update_data(train)
print("Starting test")
new_test = update(test)
print("Starting val")
new_val = update(val)

Starting train
So you mentioned before that when you wake up in the morning
to wake up awake to wake up to go for a walk gets up
fetched nodes
I'm gonna have to, you know, hear what my coworkers have to 
to know something nervous to be asked for advice to get the information to have it right
fetched nodes
So looking into one I'm going to make a mistake. That kind o
to make a mistake sad to get into trouble to apologize is none


In [ ]:
import json
with open('train_heal.json', 'w') as fp:
    json.dump(new_train, fp)

with open('test_heal.json', 'w') as fp:
    json.dump(new_test, fp)
    
with open('val_heal.json', 'w') as fp:
    json.dump(new_val, fp)

In [ ]:
input = train_df["input"].iloc[2]
x_react = eval(train_df["xReact"].iloc[2])
x_want = eval(train_df["xWant"].iloc[2])
x_need = eval(train_df["xNeed"].iloc[2])
x_intent = eval(train_df["xIntent"].iloc[2])
x_effect = eval(train_df["xEffect"].iloc[2])
print(x_react["beams"][0])
print(x_want["beams"][0])
print(x_need["beams"][0])
print(x_intent["beams"][0])
print(x_effect["beams"][0])

In [ ]:
print(1)